### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
# !pip install ipynbname

import os
import ipynbname
from IPython.display import clear_output

notebook_dir = os.path.dirname(ipynbname.path())    # notebook's dir
os.chdir(notebook_dir)

print("Current dir:", os.getcwd())


Current dir: /home/balabaevvl/courses


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

clear_output()

In [4]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [5]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()

lines = [" ".join(tokenizer.tokenize(line.lower())) for line in lines]

In [6]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$
P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}).
$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [7]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens: 
# - `UNK` represents absent tokens, 
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    n -= 1  # N-gram -> history of size N-1

    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in tqdm(lines, desc=f"Counting {n + 1}-grams", leave=False):
      tokens = [UNK] * n + line.split() + [EOS]

      for i in range(n, len(tokens)):
        history = tuple(tokens[i-n:i])

        counts[history][tokens[i]] += 1

    return counts


In [8]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [9]:
class NGramLanguageModel:    
    def __init__(self, lines, n):
        """ 
        Train a simple count-based language model: 
        compute probabilities P(w_t | prefix) given ngram counts
        
        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n
    
        counts = count_ngrams(lines, self.n)

        self.probs = self._init_probs(counts)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)
        

    def _init_probs(self, counts):
        probs = defaultdict(dict)

        for history in tqdm(counts.keys(), desc="Calculating probabilities", leave=False):
            divisor = sum(counts[history].values())
            
            for k, v in counts[history].items():
                probs[history][k] = v / divisor

        return probs

    def get_possible_next_tokens(self, prefix) -> dict[str, float]:
        """
        :param prefix: list[str] | str with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        if not isinstance(prefix, list):
            prefix = prefix.split()

        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]
    
    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [10]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [11]:
lm = NGramLanguageModel(lines, n=3)

Counting 3-grams:   0%|                                                                                                                                              | 0/41000 [00:00<?, ?it/s]

The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [12]:
import random

def get_next_token(lm: NGramLanguageModel, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    at_prefix = lm.get_possible_next_tokens(prefix)
    
    if temperature == 0:
        return max(at_prefix, key=at_prefix.get)

    tokens, probs = zip(*at_prefix.items())
    probs = list(map(lambda x: x ** (1 / temperature), probs))

    token = random.choices(tokens, weights=probs, k=1)

    return token[0]

In [13]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [14]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial mutation inspired hyper - parameters . we start by comparing with a predictive representation of mcpss . we augment the loss of 0 . 80 % of all such tasks . we conclude by raising a few principal components and their competences to perform target reaching after training , the activity of thousands of nodes between layers that learn certain classes than for chinese characters . next , we provide an efficient method for analysing large number of key types of random geometric graphs is to explain our language model to estimate action progress on an unsupervised neural network lms


In [15]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

bridging the gap between the two - stage approach to the best of our approach is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand , the proposed method is based on the other hand ,


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [16]:
def perplexity(lm, token_lines, min_logprob=np.log(10 ** -50.)):
    """
    :param token_lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above
    
    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)
    
    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    loglikelihood = 0
    T = 0

    for line in tqdm(token_lines, desc="Calculating Perplexity", leave=False):
        tokens = [UNK] + line.split() + [EOS]

        size = len(tokens)
        T += size - 1   # including EOS, excluding UNK

        for i in range(1, size):
            loglikelihood += np.log(lm.get_next_token_prob(tokens[:i], tokens[i]) + 1e-10) 

    perplexity = np.exp(-loglikelihood / T)
    
    return perplexity

In [17]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [18]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


Counting 1-grams:   0%|                                                                                                                                              | 0/30750 [00:00<?, ?it/s]

N = 1, Perplexity = 943.03154


N = 2, Perplexity = 848.97732


N = 3, Perplexity = 125108.96164


In [19]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [20]:
class LaplaceLanguageModel(NGramLanguageModel): 
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.vocab_size = len(self.vocab)
        self.probs = defaultdict(dict)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * self.vocab_size

            for token in token_counts:
                self.probs[prefix][token] = (token_counts[token] + delta) / total_count


    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)

        missing_prob_total = 1.0 - sum(token_probs.values())
        # print(missing_prob_total)
        missing_prob = missing_prob_total / max(1, self.vocab_size - len(token_probs))

        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    
    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)

        if next_token in token_probs:
            return token_probs[next_token]

        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob_total = max(0, missing_prob_total) # prevent rounding errors

        return missing_prob_total / max(1, self.vocab_size - len(token_probs))
        

**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [21]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [22]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

Counting 1-grams:   1%|▊                                                                                                                                 | 196/30750 [00:00<00:15, 1954.93it/s]

N = 1, Perplexity = 943.20974


N = 2, Perplexity = 470.47881


N = 3, Perplexity = 3679.43788


In [23]:
prefix = "attention is all you need" # <- your ideas :)

for i in range(20):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

attention is all you need tivariate decay opqtable boyan sachs proliferative disrupting pronounciations archetypal assembles fundmamental subtlety stereo retino rbcs cerebellar concomitant paso astoundingly 2202


### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formula.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [24]:
import math

In [25]:
class KneserNeyLanguageModel(NGramLanguageModel): 
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        assert n >= 1
        self.n = n
        self.delta = delta

        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.vocab_size = len(self.vocab)

        self.probs = self._init_probs(lines)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)

    def _init_probs(self, lines):
        level_probs: dict[int, dict[tuple[str], int]] = {}

        cont_counts = self._calculate_continuation_counts(lines)
        level_probs[1] = self._calculate_continuation_probs(cont_counts)

        for level in tqdm(range(2, self.n + 1), desc="Processing N-Grams", leave=False):
            counts = count_ngrams(lines, level)

            level_probs[level] = self._calculate_level_probs(counts)

            lambdas = self._calculate_level_normalization_weight_lambda(counts)

            for history in tqdm(counts.keys(), desc=f"Adding {level}-Gram", leave=False):
                current_level_probs = level_probs[level][history]
                lambda_history = lambdas[history]
                shorter_history = history[1:]
                previous_level_probs = level_probs[level - 1][shorter_history]

                # for w in self.vocab:
                for w in counts[history].keys():
                    current_level_probs[w] = current_level_probs.get(w, 0) + lambda_history * previous_level_probs[w]

        return level_probs[self.n]


    def _calculate_continuation_counts(self, lines):
        cont_counts_set = defaultdict(set)

        for line in tqdm(lines, desc="Continuation Counting", leave=False):
            tokens = [UNK] + line.split() + [EOS]

            for i in range(1, len(tokens)):
                cont_counts_set[tokens[i]].add(tokens[i - 1])

        cont_counts = defaultdict(int)
        for key, s in cont_counts_set.items():
            cont_counts[key] = len(s)

        return cont_counts

    def _calculate_continuation_probs(self, cont_counts):
        probs = defaultdict(dict)
        
        a = tuple()
        divisor = sum(cont_counts.values())
        for k, v in cont_counts.items():
            probs[a][k] = v / divisor

        return probs
        

    def _calculate_level_probs(self, counts):
        probs = defaultdict(dict)

        for history in tqdm(counts.keys(), desc="Calculating probabilities", leave=False):
            history_occurances = sum(counts[history].values())

            for k, v in counts[history].items():
                probs[history][k] = max(0, v - self.delta) / history_occurances

        return probs
    

    def _calculate_level_normalization_weight_lambda(self, counts):
        lambdas = defaultdict(float)
        
        for history in tqdm(counts.keys(), desc="Calculating normalization weights", leave=False):
            history_occurances = sum(counts[history].values())
            distinct_succsessors = sum(map(lambda x: int(x > 0), counts[history].values()))

            lambdas[history] = (self.delta * distinct_succsessors) / history_occurances
            
        return lambdas


    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)

        missing_prob_total = 1.0 - sum(token_probs.values())
        
        if math.isclose(missing_prob_total, 0):
            return token_probs

        missing_prob = missing_prob_total / max(1, self.vocab_size - len(token_probs))

        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    
    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)

        if next_token in token_probs:
            return token_probs[next_token]

        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob_total = max(0, missing_prob_total)

        return missing_prob_total / max(1, self.vocab_size - len(token_probs))


In [26]:
# n_grams = count_ngrams(train_lines, n=3)

In [28]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [36]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

N = 1, Perplexity = 1485.24275


N = 2, Perplexity = 259.48779


N = 3, Perplexity = 959.44042
